In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q wfdb

import os
import sys

# Set the working directory to your project folder
project_path = '/content/drive/MyDrive/FYP'
if os.path.exists(project_path):
    os.chdir(project_path)
    print(f"Working directory changed to: {os.getcwd()}")
else:
    print(f"Warning: {project_path} not found.")

sys.path.insert(0, os.getcwd())

import numpy as np, pandas as pd, torch
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from src import config as C
from src.data import load_beats
from src.segment import build_dataset
from src.train2 import fit, get_logits, DEVICE
print('device:', DEVICE)

Working directory changed to: /content/drive/MyDrive/FYP
device: cuda


In [3]:
df = load_beats()
d1 = df[df.split == 'DS1']
X, RR, y, rec, ridx = build_dataset(d1)
print(X.shape, RR.shape, np.bincount(y))

(53138, 180) (53138, 4) [47917   999  3807   415]


In [4]:
FOLDS = {
    0: [109, 118, 119, 115, 230],
    1: [207, 106, 202, 112, 122],
    2: [208, 124, 114, 116],
    3: [203, 209, 101, 108],
    4: [201, 205, 215, 220, 223],
}
rec2fold = {r: k for k, rs in FOLDS.items() for r in rs}
assert sorted(rec2fold) == sorted(set(int(r) for r in rec))   # every DS1 record exactly once

oof = np.zeros((len(X), 4), dtype=np.float32)
fold_id = np.array([rec2fold[int(r)] for r in rec])
for k in range(5):
    va = np.where(fold_id == k)[0]
    tr = np.where(fold_id != k)[0]
    print(f'fold {k}: val {FOLDS[k]} | train F {(y[tr]==3).sum()}, S {(y[tr]==1).sum()}, V {(y[tr]==2).sum()}')
    m = fit(X[tr], RR[tr], y[tr])
    oof[va] = get_logits(m, X[va], RR[va])

fold 0: val [109, 118, 119, 115, 230] | train F 413, S 903, V 3308
  epoch 1: loss 0.5584
  epoch 2: loss 0.2952
  epoch 3: loss 0.2382
  epoch 4: loss 0.1985
  epoch 5: loss 0.1819
  epoch 6: loss 0.1711
  epoch 7: loss 0.1519
  epoch 8: loss 0.1496
  epoch 9: loss 0.1419
  epoch 10: loss 0.1348
  epoch 11: loss 0.1302
  epoch 12: loss 0.1221
  epoch 13: loss 0.1222
  epoch 14: loss 0.1091
  epoch 15: loss 0.1096
fold 1: val [207, 106, 202, 112, 122] | train F 414, S 835, V 3058
  epoch 1: loss 0.5360
  epoch 2: loss 0.2810
  epoch 3: loss 0.2334
  epoch 4: loss 0.1990
  epoch 5: loss 0.1798
  epoch 6: loss 0.1606
  epoch 7: loss 0.1604
  epoch 8: loss 0.1437
  epoch 9: loss 0.1419
  epoch 10: loss 0.1304
  epoch 11: loss 0.1238
  epoch 12: loss 0.1202
  epoch 13: loss 0.1126
  epoch 14: loss 0.1103
  epoch 15: loss 0.1118
fold 2: val [208, 124, 114, 116] | train F 34, S 953, V 2616
  epoch 1: loss 0.5222
  epoch 2: loss 0.2806
  epoch 3: loss 0.2298
  epoch 4: loss 0.2068
  epoch 5: 

In [5]:
pred = oof.argmax(1)
print(classification_report(y, pred, target_names=C.CLASSES, digits=3))
print(confusion_matrix(y, pred))
np.savez(C.PROC / 'oof_ds1_v3.npz', logits=oof, y=y, rec=rec, ridx=ridx, fold=fold_id)

              precision    recall  f1-score   support

           N      0.975     0.787     0.871     47917
           S      0.100     0.426     0.162       999
           V      0.422     0.841     0.562      3807
           F      0.018     0.111     0.030       415

    accuracy                          0.779     53138
   macro avg      0.379     0.541     0.406     53138
weighted avg      0.911     0.779     0.829     53138

[[37696  3702  4002  2517]
 [  360   426   208     5]
 [  428   126  3201    52]
 [  191     3   175    46]]


In [6]:
for k in range(5):
    m_ = fold_id == k
    nrec = (pred[m_][y[m_] == 0] == 0).mean()
    print(f'fold {k}: acc {(pred[m_] == y[m_]).mean():.3f} | N recall {nrec:.3f} | records {sorted(set(int(r) for r in rec[m_]))}')

r_ = pd.DataFrame({'rec': rec, 'y': y, 'pred': pred})
n_rec = r_[r_.y == 0].assign(ok=lambda d: d.pred == 0).groupby('rec').ok.mean().round(3)
print(n_rec.sort_values().to_string())

fold 0: acc 0.847 | N recall 0.848 | records [109, 115, 118, 119, 230]
fold 1: acc 0.840 | N recall 0.855 | records [106, 112, 122, 202, 207]
fold 2: acc 0.845 | N recall 0.865 | records [114, 116, 124, 208]
fold 3: acc 0.618 | N recall 0.614 | records [101, 108, 203, 209]
fold 4: acc 0.741 | N recall 0.753 | records [201, 205, 215, 220, 223]
rec
207    0.073
203    0.209
220    0.243
101    0.565
118    0.620
116    0.737
201    0.739
230    0.783
205    0.840
108    0.859
209    0.878
215    0.883
208    0.886
109    0.893
114    0.899
223    0.938
106    0.989
112    0.994
124    0.995
115    0.997
202    0.997
122    1.000
119    1.000


In [7]:
from scipy.special import softmax
pr = softmax(oof, 1); conf = pr.max(1)
for r in [109, 207, 220, 203, 101, 112]:
    m_ = (rec == r) & (y == 0)
    wrong = m_ & (pred != 0)
    print(r, 'N beats:', m_.sum(), '| mean conf all:', conf[m_].mean().round(3),
          '| mean conf when wrong:', conf[wrong].mean().round(3) if wrong.any() else '-')

109 N beats: 2491 | mean conf all: 0.747 | mean conf when wrong: 0.62
207 N beats: 1542 | mean conf all: 0.965 | mean conf when wrong: 0.966
220 N beats: 1953 | mean conf all: 0.706 | mean conf when wrong: 0.737
203 N beats: 2529 | mean conf all: 0.727 | mean conf when wrong: 0.724
101 N beats: 1859 | mean conf all: 0.662 | mean conf when wrong: 0.613
112 N beats: 2537 | mean conf all: 0.961 | mean conf when wrong: 0.567


In [ ]:
# from src.train2 import fit, get_logits, DEVICE
# from src.train2 import fit_curve
# H = []
# for k in range(5):
#     va = np.where(fold_id == k)[0]; tr = np.where(fold_id != k)[0]
#     H.append(fit_curve(X[tr], RR[tr], y[tr], X[va], RR[va], y[va], epochs=40))
# H = np.stack(H)                       # [fold, epoch, metric]
# mean = np.nanmean(H, axis=0)
# tab = pd.DataFrame(mean, columns=['val_NLL', 'acc', 'rec_N', 'rec_S', 'rec_V'],
#                    index=np.arange(1, 41))
# print(tab.iloc[[0, 2, 4, 7, 9, 14, 19, 24, 29, 34, 39]].round(3))

In [10]:
import torch, contextlib, io
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

SEEDS, EPOCHS = [42, 43, 44], 10
oofs = []
for s in SEEDS:
    o = np.zeros((len(X), 4), np.float32)
    for k in range(5):
        va = np.where(fold_id == k)[0]; tr = np.where(fold_id != k)[0]
        with contextlib.redirect_stdout(io.StringIO()):          # hide per-epoch lines
            m = fit(X[tr], RR[tr], y[tr], epochs=EPOCHS, seed=s)
        o[va] = get_logits(m, X[va], RR[va])
    oofs.append(o)
    p = o.argmax(1)
    print(f'seed {s}: acc {(p == y).mean():.3f} | recall N/S/V/F',
          [round((p[y == c] == c).mean(), 3) for c in range(4)])

np.savez(C.PROC / 'oof_ds1_v4_seeds.npz', logits=np.stack(oofs), y=y, rec=rec,
         ridx=ridx, fold=fold_id, seeds=SEEDS)

seed 42: acc 0.815 | recall N/S/V/F [np.float64(0.824), np.float64(0.382), np.float64(0.895), np.float64(0.012)]
seed 43: acc 0.784 | recall N/S/V/F [np.float64(0.802), np.float64(0.231), np.float64(0.787), np.float64(0.029)]
seed 44: acc 0.749 | recall N/S/V/F [np.float64(0.754), np.float64(0.398), np.float64(0.855), np.float64(0.005)]


In [11]:
tabs = []
for o in oofs:
    p = o.argmax(1)
    d = pd.DataFrame({'rec': rec, 'y': y, 'ok': p == y})
    tabs.append(d[d.y == 0].groupby('rec').ok.mean())
T = pd.concat(tabs, axis=1, keys=[f'seed{s}' for s in SEEDS]).round(3)
T['range'] = T.max(1) - T.min(1)
print(T.sort_values('range', ascending=False).head(10))

     seed42  seed43  seed44  range
rec                               
101   0.970   0.999   0.037  0.962
109   0.961   0.064   0.921  0.897
115   0.483   0.055   0.774  0.719
118   0.724   0.151   0.157  0.573
203   0.339   0.636   0.065  0.571
220   0.445   0.995   0.998  0.553
112   0.863   0.979   0.466  0.513
230   0.968   0.652   0.795  0.316
116   0.749   0.986   0.868  0.237
122   0.952   1.000   0.804  0.196


In [12]:
from scipy.special import softmax
P = np.mean([softmax(o, 1) for o in oofs], axis=0)       # ensemble probabilities
pe = P.argmax(1)
print('ensemble acc', round((pe == y).mean(), 3),
      '| recall N/S/V/F', [round((pe[y == c] == c).mean(), 3) for c in range(4)])

def n_recall_by_record(preds_list):
    cols = []
    for p in preds_list:
        d = pd.DataFrame({'rec': rec, 'y': y, 'ok': p == y})
        cols.append(d[d.y == 0].groupby('rec').ok.mean())
    return pd.concat(cols, axis=1)

base = n_recall_by_record([o.argmax(1) for o in oofs])
print('baseline: mean per-record range of N recall across seeds:',
      (base.max(1) - base.min(1)).mean().round(3))
print(n_recall_by_record([pe]).round(3).T.to_string())    # ensemble, per record

ensemble acc 0.832 | recall N/S/V/F [np.float64(0.848), np.float64(0.329), np.float64(0.859), np.float64(0.007)]
baseline: mean per-record range of N recall across seeds: 0.275
rec    101    106    108    109    112    114    115    116    118  119  122  124    201    202    203    205    207    208    209    215    220    223    230
ok   0.986  0.993  0.963  0.859  0.895  0.997  0.452  0.905  0.258  1.0  1.0  1.0  0.983  0.994  0.362  0.999  0.062  0.938  0.944  0.986  0.964  0.993  0.913


In [13]:
def run_cv(seeds=(42, 43, 44), epochs=10, **kw):
    outs = []
    for s in seeds:
        o = np.zeros((len(X), 4), np.float32)
        for k in range(5):
            va = np.where(fold_id == k)[0]; tr = np.where(fold_id != k)[0]
            with contextlib.redirect_stdout(io.StringIO()):
                m = fit(X[tr], RR[tr], y[tr], epochs=epochs, seed=s, **kw)
            o[va] = get_logits(m, X[va], RR[va])
        outs.append(o)
    return outs

def summarize(name, outs):
    accs = [(o.argmax(1) == y).mean() for o in outs]
    rc = np.array([[(o.argmax(1)[y == c] == c).mean() for c in range(3)] for o in outs])
    t = n_recall_by_record([o.argmax(1) for o in outs])
    Pe = np.mean([softmax(o, 1) for o in outs], axis=0).argmax(1)
    print(f'{name:10s} acc {np.mean(accs):.3f}±{np.std(accs):.3f} | recall N/S/V {rc.mean(0).round(3)} '
          f'| N-recall range/record {(t.max(1) - t.min(1)).mean():.3f} | ensemble acc {(Pe == y).mean():.3f}')

res = {}
res['aug']    = run_cv(aug=True);            summarize('aug', res['aug'])
res['aug_wd'] = run_cv(aug=True, wd=1e-3);   summarize('aug_wd', res['aug_wd'])
summarize('baseline', oofs)

aug        acc 0.789±0.027 | recall N/S/V [0.802 0.287 0.839] | N-recall range/record 0.154 | ensemble acc 0.823
aug_wd     acc 0.801±0.037 | recall N/S/V [0.816 0.297 0.828] | N-recall range/record 0.162 | ensemble acc 0.833
baseline   acc 0.782±0.027 | recall N/S/V [0.793 0.337 0.846] | N-recall range/record 0.275 | ensemble acc 0.832


In [14]:
np.savez(C.PROC / 'oof_ds1_v5_augwd.npz', logits=np.stack(res['aug_wd']), y=y, rec=rec,
         ridx=ridx, fold=fold_id, seeds=[42, 43, 44])

# final models on ALL of DS1, for the DS2 evaluation in Phase 6 (DS2 is not touched here)
for s in [42, 43, 44]:
    with contextlib.redirect_stdout(io.StringIO()):
        m = fit(X, RR, y, epochs=10, seed=s, aug=True, wd=1e-3)
    torch.save(m.state_dict(), C.PROC / f'cnn_ds1_seed{s}.pt')
print('saved')

saved
